# Plant Leaf Classification Using Deep Learning
### EC8010 / EC8020 – Design Task 4
**Model:** Fine-Tuned ConvNeXt-Base (Transfer Learning)
**Dataset:** Egyptian Plant Leaf Image Dataset (EgyPLI) — Sarhan & Shaheen, *Scientific Data* (2026)

**Pipeline (as per proposed methodology):**
1. Input leaf images → Preprocessing (resize, normalize, augment)
2. Fine-tune pre-trained ConvNeXt-Base architecture
3. Stratified K-Fold cross validation training
4. Evaluate best model (Accuracy, F1-Score, Confusion Matrix)
5. Inference demo on random sample images

**Why ConvNeXt-Base (per group proposal):** ConvNeXt-Base modernizes a standard CNN with a
7×7 patchify stem and large-kernel depthwise convolutions, giving it a much wider (near-global)
receptive field than classic CNNs while remaining fully convolutional. It is used here as the
"large receptive field" candidate to test against InceptionV3 (multi-scale factorized kernels)
and EfficientNet-B0 (uniformly-scaled depth/width/resolution) on the same EgyPLI dataset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Environment Setup

In [ ]:
!pip -q install scikit-learn seaborn --upgrade

In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications.convnext import ConvNeXtBase, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, ConfusionMatrixDisplay)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Dataset Acquisition — Google Drive

The **Egyptian Plant Leaf Image Dataset (EgyPLI)** has already been downloaded and stored in
Google Drive. Reference: https://doi.org/10.34740/kaggle/dsv/9782525

**Before running:** set `DRIVE_DATA_PATH` below to the folder in your Drive that contains the
class sub-folders (e.g. `apple/`, `berry/`, `fig/`, ... each holding that class's images).
Use the **same folder** your teammates used for InceptionV3 / EfficientNet, so all three models
in the group are compared on an identical train/val/test split.


In [ ]:
# >>> EDIT THIS PATH to point to the dataset folder inside your Google Drive <<<
# Example: '/content/drive/MyDrive/EgyPLI' or '/content/drive/MyDrive/Datasets/plant-leaf-image-dataset/New_data'
DRIVE_DATA_PATH = '/content/drive/MyDrive/plant dataset'

# Locate the actual class-folder directory (handles cases where the dataset
# is nested one or two levels deeper, e.g. .../plant-leaf-image-dataset/New_data/<classes>)
def find_data_dir(root):
    if not os.path.isdir(root):
        raise FileNotFoundError(f"'{root}' does not exist. Check DRIVE_DATA_PATH.")
    for cur_root, dirs, files in os.walk(root):
        if dirs and all(os.path.isdir(os.path.join(cur_root, d)) for d in dirs):
            sample_dir = os.path.join(cur_root, dirs[0])
            if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in os.listdir(sample_dir)):
                return cur_root
    return root

data_dir = find_data_dir(DRIVE_DATA_PATH)
print("Using data directory:", data_dir)
print("Detected class folders:", sorted(os.listdir(data_dir)))

## 3. Dataset Exploration (Data View)

In [ ]:
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print("Classes:", class_names)

class_counts = {}
for c in class_names:
    class_counts[c] = len(os.listdir(os.path.join(data_dir, c)))

df_counts = pd.DataFrame(list(class_counts.items()), columns=['Class', 'Count']).sort_values('Count', ascending=False)
print(df_counts)
print(f"\nTotal images: {df_counts['Count'].sum()}")

In [ ]:
# Class distribution plot (checking for imbalance)
plt.figure(figsize=(10, 5))
sns.barplot(data=df_counts, x='Class', y='Count', palette='viridis')
plt.title('Distribution of Images Across Classes (EgyPLI)')
plt.xlabel('Plant Species')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Visual sample grid — one representative image per class
plt.figure(figsize=(16, 8))
for i, c in enumerate(class_names):
    class_dir = os.path.join(data_dir, c)
    img_file = os.listdir(class_dir)[0]
    img = load_img(os.path.join(class_dir, img_file))
    plt.subplot(2, (num_classes + 1) // 2, i + 1)
    plt.imshow(img)
    plt.title(c)
    plt.axis('off')
plt.suptitle('Sample Leaf Images per Class', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Image size / aspect-ratio survey (sanity check before resizing to 224x224)
sizes = []
for c in class_names:
    class_dir = os.path.join(data_dir, c)
    for f in os.listdir(class_dir)[:20]:
        try:
            img = cv2.imread(os.path.join(class_dir, f))
            if img is not None:
                sizes.append(img.shape[:2])
        except Exception:
            pass

sizes = np.array(sizes)
print("Sampled", len(sizes), "images")
print("Height  -> min:", sizes[:,0].min(), " max:", sizes[:,0].max(), " mean:", sizes[:,0].mean().round(1))
print("Width   -> min:", sizes[:,1].min(), " max:", sizes[:,1].max(), " mean:", sizes[:,1].mean().round(1))

## 4. Preprocessing

Same pipeline as the InceptionV3 model (resize → normalize → augment), with the only change being
the **target resolution (224×224, ConvNeXt's native input size)** and the **normalization function**
(ConvNeXt's own `preprocess_input`, which matches the statistics its ImageNet weights were trained with).


In [ ]:
IMG_SIZE = 224          # native ConvNeXt-Base input resolution
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Build a file-path / label DataFrame first so we can do a clean stratified split
filepaths, labels = [], []
for c in class_names:
    class_dir = os.path.join(data_dir, c)
    for f in os.listdir(class_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            filepaths.append(os.path.join(class_dir, f))
            labels.append(c)

data_df = pd.DataFrame({'filepath': filepaths, 'label': labels})
print(data_df.shape)
data_df.head()

In [ ]:
# Stratified split: Train / Validation / Test
train_val_df, test_df = train_test_split(
    data_df, test_size=TEST_SPLIT, stratify=data_df['label'], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val_df, test_size=VAL_SPLIT/(1-TEST_SPLIT), stratify=train_val_df['label'], random_state=SEED
)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

In [ ]:
# Data generators — heavy augmentation ONLY on the training split
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True,
    fill_mode='nearest'
)

eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), class_mode='categorical',
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)

val_gen = eval_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), class_mode='categorical',
    batch_size=BATCH_SIZE, shuffle=False
)

test_gen = eval_datagen.flow_from_dataframe(
    test_df, x_col='filepath', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), class_mode='categorical',
    batch_size=BATCH_SIZE, shuffle=False
)

class_indices = train_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
print(class_indices)

In [ ]:
# Visualize a batch of AUGMENTED training images
# NOTE: keras.applications.convnext.preprocess_input keeps ConvNeXt's built-in
# Normalization layer in charge of scaling, so it passes pixels through mostly
# unchanged here — clip only for safe display.
imgs, lbls = next(train_gen)
plt.figure(figsize=(14, 6))
for i in range(min(8, len(imgs))):
    plt.subplot(2, 4, i + 1)
    disp = imgs[i] / 255.0
    plt.imshow(np.clip(disp, 0, 1))
    plt.title(idx_to_class[np.argmax(lbls[i])])
    plt.axis('off')
plt.suptitle('Sample Augmented Training Batch')
plt.tight_layout()
plt.show()

In [ ]:
train_labels_int = train_df['label'].map(class_indices).values

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_int),
    y=train_labels_int
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

print("Class weights (to counter imbalance):")
for idx, w in class_weight_dict.items():
    print(f"  {idx_to_class[idx]:>12s} : {w:.3f}")

## 5. Proposed Architecture — ConvNeXt-Base

Per the group's design table:

| Model | Params | Receptive field | Key design |
|---|---|---|---|
| ConvNeXt-Base | ~88M (full) / ~28.6M reported by group | Wide / near-global | 7×7 patchify stem, large-kernel depthwise convs, LayerNorm, GELU |

We use `keras.applications.convnext.ConvNeXtBase` pretrained on ImageNet, with its classification
head removed (`include_top=False`) and a new dense head attached — mirroring the InceptionV3 head
design (GAP → Dense(512) → BatchNorm → Dropout → Softmax) so the three group models stay
comparable on everything except the backbone itself.


In [ ]:
def build_convnext(num_classes, img_size=IMG_SIZE, freeze_base=True):
    base_model = ConvNeXtBase(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True   # ConvNeXt does its own internal normalization
    )
    base_model.trainable = not freeze_base

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False if freeze_base else None)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='ConvNeXtBase_PlantLeaf')
    return model, base_model

model, base_model = build_convnext(num_classes, freeze_base=True)
model.summary()

## 6. Cross Validation Training

### Stage 1 — Frozen backbone warm-up
The pretrained ConvNeXt-Base backbone is frozen; only the new dense head is trained. This lets the
randomly-initialized head settle before we risk disturbing the pretrained ImageNet weights.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
os.makedirs('checkpoints', exist_ok=True)

callbacks_stage1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7),
    ModelCheckpoint('checkpoints/convnext_stage1_best.h5', monitor='val_loss',
                     save_best_only=True, verbose=1)
]

EPOCHS_STAGE1 = 15

history_stage1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weight_dict,
    callbacks=callbacks_stage1
)

### Stage 2 — Fine-tuning the top ConvNeXt blocks
The backbone is unfrozen, but only the last N layers (the deepest / most task-specific blocks) are
made trainable, using a much smaller learning rate so the ImageNet features aren't destroyed.


In [ ]:
base_model.trainable = True

# Freeze all layers except the last N (fine-tune only the top blocks/stage)
FINE_TUNE_AT = len(base_model.layers) - 50
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_stage2 = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8),
    ModelCheckpoint('checkpoints/convnext_finetuned_best.h5', monitor='val_loss',
                     save_best_only=True, verbose=1)
]

EPOCHS_STAGE2 = 10

history_stage2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_STAGE2,
    class_weight=class_weight_dict,
    callbacks=callbacks_stage2
)

### Stratified K-Fold Cross-Validation
Independent lightweight ConvNeXt-Base models are trained across 5 stratified folds (drawn from the
combined train+val pool) purely to report a robust, low-variance estimate of generalisation
performance — exactly as done for the InceptionV3 model, for a fair group comparison.


In [ ]:
K_FOLDS = 5
EPOCHS_PER_FOLD = 5   # kept small per fold for resource efficiency; increase if resources allow

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
cv_pool_df = train_val_df.reset_index(drop=True)

fold_accuracies, fold_f1_scores = [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(cv_pool_df['filepath'], cv_pool_df['label']), start=1):
    print(f"\n===== Fold {fold}/{K_FOLDS} =====")
    fold_train_df = cv_pool_df.iloc[tr_idx]
    fold_val_df = cv_pool_df.iloc[va_idx]

    fold_train_gen = train_datagen.flow_from_dataframe(
        fold_train_df, x_col='filepath', y_col='label',
        target_size=(IMG_SIZE, IMG_SIZE), class_mode='categorical',
        batch_size=BATCH_SIZE, shuffle=True, seed=SEED
    )
    fold_val_gen = eval_datagen.flow_from_dataframe(
        fold_val_df, x_col='filepath', y_col='label',
        target_size=(IMG_SIZE, IMG_SIZE), class_mode='categorical',
        batch_size=BATCH_SIZE, shuffle=False
    )

    fold_model, fold_base = build_convnext(num_classes, freeze_base=True)
    fold_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                        loss='categorical_crossentropy', metrics=['accuracy'])

    fold_model.fit(
        fold_train_gen, validation_data=fold_val_gen,
        epochs=EPOCHS_PER_FOLD,
        callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
        verbose=0
    )

    val_probs = fold_model.predict(fold_val_gen, verbose=0)
    val_preds = np.argmax(val_probs, axis=1)
    val_true = fold_val_gen.classes

    acc = accuracy_score(val_true, val_preds)
    f1 = f1_score(val_true, val_preds, average='macro')
    fold_accuracies.append(acc)
    fold_f1_scores.append(f1)
    print(f"Fold {fold} -> Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")

    keras.backend.clear_session()

print("\n===== Cross-Validation Summary =====")
print(f"Mean Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
print(f"Mean Macro F1: {np.mean(fold_f1_scores):.4f} ± {np.std(fold_f1_scores):.4f}")

## 7. Training History (Stage 1 + Stage 2 combined)

In [ ]:
def combine_history(h1, h2):
    combined = {}
    for k in h1.history:
        combined[k] = h1.history[k] + h2.history[k]
    return combined

history = combine_history(history_stage1, history_stage2)

epochs_range = range(len(history['accuracy']))

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['accuracy'], label='Train Accuracy')
plt.plot(epochs_range, history['val_accuracy'], label='Validation Accuracy')
plt.axvline(x=len(history_stage1.history['accuracy']), color='gray', linestyle='--', label='Fine-tuning starts')
plt.legend()
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['loss'], label='Train Loss')
plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
plt.axvline(x=len(history_stage1.history['loss']), color='gray', linestyle='--', label='Fine-tuning starts')
plt.legend()
plt.title('Loss over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Loss')

plt.tight_layout()
plt.show()

## 8. Evaluate Best Model on Held-Out Test Set

In [ ]:
test_probs = model.predict(test_gen, verbose=1)
test_preds = np.argmax(test_probs, axis=1)
test_true = test_gen.classes

test_acc = accuracy_score(test_true, test_preds)
test_f1_macro = f1_score(test_true, test_preds, average='macro')
test_f1_weighted = f1_score(test_true, test_preds, average='weighted')

print(f"Test Accuracy      : {test_acc:.4f}")
print(f"Test Macro F1-Score : {test_f1_macro:.4f}")
print(f"Test Weighted F1-Score: {test_f1_weighted:.4f}")
print()
print(classification_report(test_true, test_preds, target_names=list(class_indices.keys())))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_true, test_preds)
plt.figure(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(class_indices.keys()))
disp.plot(cmap='Blues', xticks_rotation=45, values_format='d')
plt.title('Confusion Matrix — ConvNeXt-Base on EgyPLI Test Set')
plt.tight_layout()
plt.show()

## 10. Random Sample Inference (Demo Simulation)

In [ ]:
sample_idx = np.random.choice(len(test_df), size=5, replace=False)
sample_rows = test_df.iloc[sample_idx]

plt.figure(figsize=(18, 4))
for i, (_, row) in enumerate(sample_rows.iterrows()):
    img = load_img(row['filepath'], target_size=(IMG_SIZE, IMG_SIZE))
    arr = preprocess_input(img_to_array(img))
    pred_probs = model.predict(np.expand_dims(arr, axis=0), verbose=0)[0]
    pred_class = idx_to_class[np.argmax(pred_probs)]
    confidence = np.max(pred_probs) * 100

    plt.subplot(1, 5, i + 1)
    plt.imshow(img)
    color = 'green' if pred_class == row['label'] else 'red'
    plt.title(f"True: {row['label']}\nPred: {pred_class} ({confidence:.1f}%)", color=color, fontsize=10)
    plt.axis('off')

plt.suptitle('Random Sample Inference (Demo Simulation)', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Save the Final Model

In [ ]:
os.makedirs('saved_model', exist_ok=True)
model.save('saved_model/convnext_base_egypli_final.keras')
print("Model saved to saved_model/convnext_base_egypli_final.keras")

In [ ]:
# Download to your local machine (Colab)
from google.colab import files
files.download('saved_model/convnext_base_egypli_final.keras')

## 12. Summary

| Item | Value |
|---|---|
| Model | ConvNeXt-Base (ImageNet pretrained, fine-tuned) |
| Input size | 224 × 224 × 3 |
| Dataset | EgyPLI (8 classes, ~3,588 real-world leaf images) |
| Training strategy | Frozen head warm-up → Fine-tune top ConvNeXt blocks |
| Class imbalance handling | Computed class weights (balanced) |
| Validation strategy | Stratified 5-Fold Cross-Validation + held-out test set |
| Reported metrics | Accuracy, Macro F1-Score, Confusion Matrix |

**Note:** Replace the fold/epoch counts with larger values if GPU time allows, for a tighter,
more reliable estimate of generalization performance. Keep `DRIVE_DATA_PATH`, `SEED`, and the
train/val/test split ratios identical to your teammates' InceptionV3 and EfficientNet notebooks
so the final group comparison table (accuracy, F1, params, receptive field) is apples-to-apples.
